# Pix2Struct AI2D-base — DIMER diagram multiple-choice question answering tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/pix2struct-ai2d-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/pix2struct-ai2d-pipeline/blob/main/tutorials/pix2struct_ai2d_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fpix2struct--ai2d--base-ffcc4d?style=flat)](https://huggingface.co/google/pix2struct-ai2d-base) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fpix2struct-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/pix2struct) [![arXiv](https://img.shields.io/badge/arXiv-2210.03347-b31b1b.svg)](https://arxiv.org/abs/2210.03347)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** Diagram multiple-choice question answering — one diagram image plus one question with 2–6 answer options → the generated option text and its matched index — using the pinned `google/pix2struct-ai2d-base` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/pix2struct_ai2d_pipeline/pipeline.py` at revision `0bb646659f4f`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `0d6b2606efe05c77c1d0670647740802a9e68eef` (~569 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the Pix2Struct image-encoder/text-decoder (a ViT-style encoder over variable-resolution 16×16 patches and a 12-layer text decoder, 282M parameters, pretrained by parsing masked web screenshots into HTML and fine-tuned on AI2D, a set of school-science diagrams with multiple-choice questions) reads the **question and its numbered options rendered as a text header above the diagram** — the Pix2Struct convention for visual question answering, with the pinned README's `<question> (1) <a> (2) <b> …` prompt format — scales the composite to fill at most 2048 patches, and generates the answer text token by token; the carried module then matches that text to the options by normalised exact match. Decoding is greedy (`do_sample=False`) under a caller-owned `max_new_tokens` budget. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried module adds snapshot verification, the input contract (image side ceilings, a non-empty question up to 256 characters, 2–6 distinct options up to 64 characters each, the token budget), a fixed output contract, an offline header font (Pillow's bundled Aileron replaces the Hub font the upstream processor would otherwise download), and the `format_prompt`, `match_option`, `validate_inputs` and `evaluation_report` helpers. The default sample is a flat cartoon plant diagram drawn in code with six numbered markers and ten authored questions, so `accuracy` against the chance baseline is demonstration (plumbing) evidence for one drawing, not an AI2D benchmark — and the model gets six of the ten wrong, which the notebook keeps and explains.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw a synthetic labelled diagram with authored multiple-choice questions (or upload your own diagram and write your own questions) and validate it into an input manifest, choose a token budget, run the supported task, read the answers correctly (generated text matched to an option or to none, no score, a `truncated` flag), exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `accuracy`, `unmatched_rate` and a chance baseline only when the correct options are known and `not-measurable` otherwise, and export the answers, the annotated diagram and provenance.

**This notebook does not demonstrate:** Free-form or open-ended answers (the model generates text, but the contract matches it to the supplied options and reports `null` otherwise), document or scene question answering (separate checkpoints), reading a diagram's text back as a transcript, answer localisation, batch throughput, sampling or beam search, evaluation on the AI2D benchmark (not bundled; only authored questions on a drawn diagram are scored here), and any training. The model was fine-tuned on textbook-style science diagrams with numbered label markers; flat cartoons, photographs, charts and non-English questions are outside what this notebook measures, and a fluent wrong answer carries no signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both (the checkpoint is stored in bfloat16 and upcast at load). CPU is adequate: the repository's model card records 5.2 s to load and 2.1–2.4 s per question on the 640×640 drawn diagram in the Windows venv (Intel Core Ultra 9 275HX). The pinned `torch==2.14.0` install and the 565 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what an encoder–decoder model's generated tokens are; what accuracy against a chance baseline does and does not show on ten questions; that a confident answer is not a correct one.
- **Data:** the default sample is a deterministic 640×640 cartoon plant diagram drawn in code (flower, two leaves, stem, roots, soil, sun) with six numbered markers in Pillow's bundled font and ten authored multiple-choice questions with their correct options, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one image decodable by Pillow (PNG/JPEG/WebP and similar) of a **single diagram**, any colour mode, sides between 16 and 4096 px, plus your own questions typed into the form field as `question | option | option | …` lines. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/pix2struct-ai2d-base` snapshot (~569 MB in total) at revision `0d6b2606efe0…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'pix2struct-ai2d-pipeline',
    'repository_revision': '0bb646659f4f0e58fc26c3ec2917f4534083d63f',
    'embedded_module': 'src/pix2struct_ai2d_pipeline/pipeline.py',
    'embedded_modules': ['src/pix2struct_ai2d_pipeline/pipeline.py'],
    'module_sha256': 'd49f38053854bc247a6625a3b5a8a7546c5faaf3e857709dbcd01b45e4153d9c',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/pix2struct_ai2d_pipeline/` @ `0bb646659f4f`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/pix2struct_ai2d_pipeline/pipeline.py`

In [ ]:
"""Diagram multiple-choice question answering with the pinned ``google/pix2struct-ai2d-base`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Pix2Struct architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed. The question and its numbered
options are rendered as a text header on top of the diagram (the Pix2Struct VQA input convention) with
Pillow's bundled font, so no font is fetched from the Hub at inference time; the generated answer text
is matched to the options by normalised exact match.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image, ImageFont

MODEL_ID = "google/pix2struct-ai2d-base"
MODEL_REVISION = "0d6b2606efe05c77c1d0670647740802a9e68eef"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "pix2struct-ai2d-base"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Generation ceilings. AI2D answers are one option's text (the checkpoint's text_config max_length is
# 20); the default leaves room for a long option, the ceiling bounds runaway generation.
MAX_NEW_TOKENS = 64
DEFAULT_MAX_NEW_TOKENS = 16
DECODING = "greedy"
# Prompt ceilings. The question and the numbered options are rendered as header lines (wrapped at 80
# characters by the processor) above the diagram; a long prompt shrinks the diagram's share of the
# patch budget. AI2D questions carry four options; the ceiling allows a few more.
MAX_QUESTION_CHARS = 256
MAX_OPTION_CHARS = 64
MIN_OPTIONS = 2
MAX_OPTIONS = 6
# Input ceilings. The processor extracts at most MAX_PATCHES 16x16 patches (preprocessor_config.json)
# after scaling the image to fill that budget, so pixel count only guards memory during resizing.
MAX_PATCHES = 2048
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def header_font_bytes() -> bytes:
    """Pillow's bundled Aileron Regular (CC0) as TrueType bytes: the header font for the rendered question.

    The upstream image processor otherwise fetches ``ybelkada/fonts/Arial.TTF`` from the Hub at
    inference time — an unpinned, unlisted download of a proprietary font. The bundled subset covers
    the printable ASCII range, which is what a question is expected to use.
    """
    font = ImageFont.load_default(size=36)
    data = getattr(font, "font_bytes", None)
    if not data:
        raise RuntimeError("Pillow's bundled TrueType font is unavailable (FreeType support missing)")
    return bytes(data)


def normalize_answer(text: str) -> str:
    """Normalisation for option matching: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def format_prompt(question: str, options: Sequence[str]) -> str:
    """The pinned README's AI2D prompt convention: ``<question> (1) <a> (2) <b> ...``."""
    return " ".join([question] + [f"({index}) {option}" for index, option in enumerate(options, start=1)])


def match_option(answer: str, options: Sequence[str]) -> int | None:
    """Zero-based index of the option the normalised answer equals, else ``None`` (no fuzzy match)."""
    pred = normalize_answer(answer)
    for index, option in enumerate(options):
        if pred == normalize_answer(option):
            return index
    return None


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one diagram image as PIL.Image.Image (any mode, converted to RGB) plus one question string and "
        "MIN_OPTIONS..MAX_OPTIONS answer options"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "question_chars": [1, MAX_QUESTION_CHARS],
    "options": [MIN_OPTIONS, MAX_OPTIONS],
    "option_chars": [1, MAX_OPTION_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False), deterministic on a fixed device and dtype",
    "preprocessing": (
        "the question and the numbered options are rendered as a black-on-white header (Pillow's bundled "
        "font, wrapped at 80 characters) above the diagram; the composite is scaled to fill at most "
        "MAX_PATCHES 16x16 patches (aspect ratio preserved), normalised per image and flattened into patch "
        "tokens with row/column positions; the decoder generates the answer text, which is matched to the "
        "options by normalised exact match"
    ),
    "output": "one answer string (the model's decoded text) plus the matched option index or null; no score",
}


def _check_inputs(
    image: Any, question: Any, options: Any, max_new_tokens: Any
) -> tuple[Image.Image, str, list[str], int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``answer`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    if not isinstance(question, str):
        raise TypeError("question must be a str")
    checked_question = " ".join(question.split())
    if not checked_question:
        raise ValueError("question must contain at least one non-whitespace character")
    if len(checked_question) > MAX_QUESTION_CHARS:
        raise ValueError(
            f"question has {len(checked_question)} chars > MAX_QUESTION_CHARS {MAX_QUESTION_CHARS}"
        )
    if isinstance(options, str) or not isinstance(options, Sequence):
        raise TypeError("options must be a sequence of str")
    if not MIN_OPTIONS <= len(options) <= MAX_OPTIONS:
        raise ValueError(f"options must have MIN_OPTIONS={MIN_OPTIONS}..MAX_OPTIONS={MAX_OPTIONS} entries")
    checked_options = []
    for option in options:
        if not isinstance(option, str):
            raise TypeError("each option must be a str")
        checked = " ".join(option.split())
        if not checked:
            raise ValueError("each option must contain at least one non-whitespace character")
        if len(checked) > MAX_OPTION_CHARS:
            raise ValueError(f"option has {len(checked)} chars > MAX_OPTION_CHARS {MAX_OPTION_CHARS}")
        checked_options.append(checked)
    if len({normalize_answer(option) for option in checked_options}) != len(checked_options):
        raise ValueError("options must be distinct after normalisation")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, checked_question, checked_options, max_new_tokens


def validate_inputs(
    image: Image.Image,
    questions: Sequence[Mapping[str, Any]],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    ``questions`` holds mappings with ``question`` and ``options``; each is checked exactly as
    ``answer`` would check it. Rejection is reported by raising, and a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    if isinstance(questions, (str, Mapping)) or not isinstance(questions, Sequence) or not questions:
        raise TypeError("questions must be a non-empty sequence of {question, options} mappings")
    checked = []
    for entry in questions:
        if not isinstance(entry, Mapping) or "question" not in entry or "options" not in entry:
            raise TypeError("each questions entry must be a mapping with 'question' and 'options'")
        _, question, options, _ = _check_inputs(image, entry["question"], entry["options"], max_new_tokens)
        checked.append({"question": question, "options": options, "prompt": format_prompt(question, options)})
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (answer takes one diagram)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "questions": checked,
        "generation": {"max_new_tokens": int(max_new_tokens), "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    correct: Sequence[int] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``correct`` (the zero-based index of the correct option per result, in order) the report
    carries the ``accuracy`` (matched option equals the correct index; an unmatched answer counts as
    wrong), the chance baseline, the rate of unmatched answers and one per-question entry, verdict
    ``sample-sanity``; without it the report is ``not-measurable`` and says what labelled data would
    make the task measurable.
    """
    if not results:
        raise ValueError("results must contain at least one answer result")
    base = {
        "task": "diagram image + multiple-choice question -> option text (AI2D-style diagram QA)",
        "score_semantics": (
            "the answer is generated text and carries no score, probability or correctness signal; the "
            "option index is a normalised exact match of that text against the options and is null when "
            "the text matches none of them. Greedy decoding makes the output reproducible on a fixed device "
            "and dtype, a reproducibility property, not a quality one"
        ),
        "sample_kind": sample_kind,
        "n_questions": len(results),
        "truncated": [bool(result.get("truncated")) for result in results],
        "unmatched": [result.get("choice_index") is None for result in results],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if correct is None:
        return {
            **base,
            "metrics": [],
            "baselines": [],
            "verdict": "not-measurable",
            "reason": "no correct-option labels were supplied for the evaluated questions",
            "needs": (
                "multiple-choice question/answer pairs on diagrams from the deployment domain (AI2D-style "
                "annotations with the correct option marked) scored with accuracy; no such labelled set "
                "ships with this repository"
            ),
        }
    if len(correct) != len(results):
        raise ValueError(f"correct has {len(correct)} entries for {len(results)} results")
    per_question = []
    for result, gold in zip(results, correct, strict=True):
        options = list(result.get("options") or [])
        if isinstance(gold, bool) or not isinstance(gold, int) or not 0 <= gold < len(options):
            raise ValueError("each correct entry must be a valid zero-based option index for its result")
        predicted = result.get("choice_index")
        per_question.append(
            {
                "question": result.get("question"),
                "prediction": result.get("answer"),
                "choice_index": predicted,
                "correct_index": gold,
                "correct_option": options[gold],
                "correct": predicted == gold,
            }
        )
    n_options = [len(result.get("options") or []) for result in results]
    chance = sum(1.0 / n for n in n_options) / len(n_options)
    return {
        **base,
        "metrics": [
            {
                "id": "accuracy",
                "value": sum(entry["correct"] for entry in per_question) / len(per_question),
                "normalisation": "answer text lower-cased, punctuation removed, whitespace collapsed; "
                "exact match against the options; unmatched counts as wrong",
                "estimation": f"{len(per_question)} question(s) on one diagram, no dispersion estimate",
            },
            {
                "id": "unmatched_rate",
                "value": sum(entry["choice_index"] is None for entry in per_question) / len(per_question),
                "estimation": f"{len(per_question)} question(s), answers matching no option",
            },
        ],
        "baselines": [{"id": "chance", "value": chance, "note": "mean of 1/n_options over the questions"}],
        "per_question": per_question,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_question)} authored question(s) on one tutorial diagram you drew yourself; plumbing "
            "evidence, not an AI2D benchmark"
        ),
        "needs": (
            "a labelled multiple-choice set on diagrams from the deployment domain (subject, drawing style, "
            "label density) for any accuracy claim; the AI2D benchmark is not bundled"
        ),
    }


@dataclass
class Pix2StructAI2DPipeline:
    """``_runner(image, prompt, max_new_tokens)`` returns ``{"answer": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructAI2DPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        font_bytes = header_font_bytes()
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Pix2StructProcessor.from_pretrained(location, **common)
        if not getattr(processor.image_processor, "is_vqa", False):
            raise RuntimeError("snapshot image processor is not the VQA variant (is_vqa=False); refusing")
        # The checkpoint is stored in bfloat16; it is upcast to float32 for CPU inference.
        model = Pix2StructForConditionalGeneration.from_pretrained(location, dtype=torch.float32, **common)
        model = model.eval().to(resolved_device)

        def runner(image: Image.Image, prompt: str, max_new_tokens: int) -> dict[str, Any]:
            # The image processor is called directly: Pix2StructProcessor.__call__ drops the
            # font_bytes kwarg, and font_bytes is what replaces the default Hub font download
            # (see header_font_bytes). The VQA processor renders the prompt as the header.
            inputs = processor.image_processor(
                image, header_text=prompt, return_tensors="pt", font_bytes=font_bytes
            ).to(resolved_device)
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            # Encoder-decoder: the output holds only decoder tokens (decoder_start + answer + eos).
            answer_ids = generated[0]
            decoded = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
            return {"answer": decoded, "new_tokens": int(answer_ids.shape[0]) - 1}

        return cls(runner, resolved_device, "float32", source)

    def answer(
        self,
        image: Image.Image,
        question: str,
        options: Sequence[str],
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Answer one multiple-choice question about one diagram; ``answer`` is the decoded text, stripped."""
        rgb, checked_question, checked_options, checked_tokens = _check_inputs(
            image, question, options, max_new_tokens
        )
        prompt = format_prompt(checked_question, checked_options)
        raw = self._runner(rgb, prompt, checked_tokens)
        if not isinstance(raw, dict) or "answer" not in raw:
            raise RuntimeError("runner must return a dict with 'answer'")
        new_tokens = int(raw.get("new_tokens", 0))
        text = str(raw["answer"]).strip()
        return {
            "answer": text,
            "choice_index": match_option(text, checked_options),
            "question": checked_question,
            "options": checked_options,
            "prompt": prompt,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `0d6b2606efe0…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Pix2StructAI2DPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "pix2struct-ai2d-base",
  "modelId": "google/pix2struct-ai2d-base",
  "revision": "0d6b2606efe05c77c1d0670647740802a9e68eef",
  "files": [
    {
      "path": "README.md",
      "bytes": 7085,
      "sha256": "1af9a1edb7519ba201f476cf255802a3fa163e081e3014140f0293e1ab701d5f"
    },
    {
      "path": "config.json",
      "bytes": 4889,
      "sha256": "bbb61790293483693c82c55f4c41c8c79514297dae72c2454d85b8cdf2e21d2d"
    },
    {
      "path": "model.safetensors",
      "bytes": 564606744,
      "sha256": "652c92f8b995d9fc5ea46350c854fd88767bf36c391552f126f6e7487b0f27a3"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 249,
      "sha256": "9662520ca2d4e38fccb8465ea3b443a88ced7079e6cb2b6e0230bae98c83b5c1"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2201,
      "sha256": "5c87151ef0f72a99d1f766a4c418bd2a1f90aaa30a8e22fe5eca9641daebb64f"
    },
    {
      "path": "spiece.model",
      "bytes": 851388,
      "sha256": "7fd650335add59bed55a432186ca0437a09e185c2d241faab468a538fe6bcf94"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3265159,
      "sha256": "0af109b23840545ef2c286073f4373959badba1faa73c8557881d5126f6287c9"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2583,
      "sha256": "5fdb6767a49aca48fdfa43d0279321918185fc4997bdb3ea72bf3a6301a1b43d"
    }
  ],
  "totalBytes": 568740298
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Pix2StructAI2DPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic diagram or optional BYOD

The default sample is **synthetic** and carries its own references: a cartoon plant — a pink flower, two leaves, a stem, four roots in brown soil and a sun — is drawn with Pillow at 640×640 with six numbered markers in white boxes, each joined to its part by a line (the AI2D convention: numbers on the diagram, words only in the options), the same drawing the repository's smoke run used. Ten questions are authored against it (six `What does the label N represent?` questions and four about the plant), each with its correct option; they are the references for the `accuracy` sanity check later. They are not a labelled dataset, so nothing here is an AI2D measurement — and the smoke run answered only four of them correctly (chance is one in four): the model answers `root` to almost any label question about this cartoon, which the notebook keeps as a recorded finding rather than tuning the drawing until it passes. The image digest is printed for the record; it depends on the Pillow build's bundled font rendering. BYOD is optional and disabled by default; when enabled, upload one diagram and type your questions — no correct options are known for them, so the evaluation report will be `not-measurable`.

The token budget is a **caller-owned request parameter**: `max_new_tokens` bounds the answer (`DEFAULT_MAX_NEW_TOKENS = 16` fits any option; `MAX_NEW_TOKENS = 64` is the ceiling). Nothing is validated in this cell — the next section hands the image and the questions to the pipeline's own validation stage, which is the only checker. Look for a dictionary naming the sample kind, the image size and digest, the budget and the number of questions.

In [ ]:
import hashlib
import io
import math

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
byod_questions = 'What does the label 1 represent? | flower | leaf | stem | root'  # @param {type:"string"}
max_new_tokens = 16  # @param {type:"integer"}


def synthetic_diagram(size=640):
    """A cartoon plant diagram with six numbered markers; returns image + [(question, options, correct index)]."""
    image = Image.new('RGB', (size, size), 'white')
    d = ImageDraw.Draw(image)
    marker_font = ImageFont.load_default(size=34)
    d.rectangle([0, 440, 640, 640], fill=(160, 120, 70))  # soil
    d.ellipse([500, 40, 600, 140], fill=(255, 215, 0))  # sun
    d.rectangle([310, 220, 330, 440], fill=(40, 140, 40))  # stem
    d.polygon([(310, 330), (220, 290), (240, 350)], fill=(50, 170, 50))  # left leaf
    d.polygon([(330, 380), (420, 340), (400, 400)], fill=(50, 170, 50))  # right leaf
    for k in range(6):  # petals
        a = math.radians(60 * k)
        cx, cy = 320 + 45 * math.cos(a), 190 + 45 * math.sin(a)
        d.ellipse([cx - 22, cy - 22, cx + 22, cy + 22], fill=(230, 60, 120))
    d.ellipse([298, 168, 342, 212], fill=(255, 200, 40))  # flower centre
    for dx in (-60, -20, 25, 70):  # roots
        d.line([(320, 440), (320 + dx, 560)], fill=(120, 80, 40), width=5)
    markers = [('1', (140, 150), (275, 175)), ('2', (90, 330), (225, 320)), ('3', (470, 250), (332, 300)), ('4', (140, 560), (290, 520)), ('5', (550, 175), (550, 140)), ('6', (560, 600), (560, 600))]
    for text, pos, tip in markers:
        if pos != tip:
            d.line([pos, tip], fill='black', width=3)
        d.rectangle([pos[0] - 22, pos[1] - 22, pos[0] + 22, pos[1] + 22], fill='white', outline='black', width=2)
        d.text(pos, text, fill='black', font=marker_font, anchor='mm')
    qa = [
        ('What does the label 1 represent?', ['flower', 'leaf', 'stem', 'root'], 0),
        ('What does the label 2 represent?', ['flower', 'leaf', 'stem', 'root'], 1),
        ('What does the label 3 represent?', ['root', 'stem', 'leaf', 'flower'], 1),
        ('What does the label 4 represent?', ['stem', 'flower', 'root', 'leaf'], 2),
        ('What does the label 5 represent?', ['moon', 'sun', 'cloud', 'rain'], 1),
        ('What does the label 6 represent?', ['water', 'air', 'soil', 'rock'], 2),
        ('Which part of the plant is below the soil?', ['flower', 'leaf', 'stem', 'root'], 3),
        ('What provides light to the plant?', ['soil', 'sun', 'root', 'leaf'], 1),
        ('Which part connects the roots to the flower?', ['leaf', 'soil', 'stem', 'sun'], 2),
        ('Which part is at the top of the plant?', ['root', 'stem', 'flower', 'soil'], 2),
    ]
    return image, qa


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    image_name = next(iter(uploaded))
    image = Image.open(io.BytesIO(uploaded[image_name]))
    image.load()
    questions = []
    for line in byod_questions.splitlines():
        parts = [part.strip() for part in line.split('|') if part.strip()]
        if len(parts) >= 3:
            questions.append({'question': parts[0], 'options': parts[1:]})
    correct = None
    sample_kind = 'BYOD'
else:
    # Deterministic drawing: no randomness, so no seed is needed; the digest depends on the Pillow build's bundled font.
    image, qa = synthetic_diagram()
    questions = [{'question': q, 'options': options} for q, options, _ in qa]
    correct = [gold for _, _, gold in qa]
    image_name = 'synthetic_plant_diagram_640x640.png'
    sample_kind = 'synthetic'

image_sha256 = hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': image_name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': image_sha256, 'max_new_tokens': max_new_tokens, 'n_questions': len(questions), 'has_labels': correct is not None})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `answer` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, each question a non-empty string of at most `MAX_QUESTION_CHARS` characters (whitespace collapsed), `MIN_OPTIONS`..`MAX_OPTIONS` distinct non-empty options of at most `MAX_OPTION_CHARS` characters, and `max_new_tokens` in `[1, MAX_NEW_TOKENS]` — and returns an **input manifest** naming the schema (including the header-rendering preprocessing and the decoding rule), the input's observed mode and size, each checked question with its options and the exact prompt that will be rendered, the budget and the verdict. The manifest is written to `outputs/pix2struct_ai2d_input_manifest.json`. To show what rejection looks like, the cell also validates a question with a single option and records the pipeline's own error message as a finding. Inside the pipeline the image is converted to RGB, the prompt is rendered above it, and the composite is scaled to the patch budget; nothing else is dropped or altered. The pipeline cannot tell whether the image is a diagram or whether the question is answerable from it: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PATCHES': MAX_PATCHES, 'MAX_QUESTION_CHARS': MAX_QUESTION_CHARS, 'MIN_OPTIONS': MIN_OPTIONS, 'MAX_OPTIONS': MAX_OPTIONS, 'MAX_OPTION_CHARS': MAX_OPTION_CHARS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING}})
input_manifest = validate_inputs(image, questions, max_new_tokens=max_new_tokens, names=[image_name])
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(image, [{'question': 'What is this?', 'options': ['a plant']}])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'single-option-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/pix2struct_ai2d_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Answer the questions and read the output correctly

`answer` returns, per question, a dict with `answer` (the decoded text, stripped), `choice_index` (the zero-based option the normalised text equals, or `null` when it equals none — there is no fuzzy match), the checked `question` and `options`, the rendered `prompt`, `image_size`, `new_tokens`, a `truncated` flag that is true when the budget was exhausted, the generation settings and the model identity. **No score exists**: the answer is generated text with no probability and no correctness signal, and matching an option is not evidence that it is the right one. Greedy decoding is deterministic on a fixed device and dtype; CUDA kernel selection can change a token and therefore the rest of the answer, so GPU and CPU outputs need not match. Each call renders the prompt as a header and re-encodes the diagram, so cost is per question (about 2.1–2.4 s each on the reference CPU). As recorded in the model card, the repository's CPU smoke on this same drawing answered `root` to the label questions 1–4 whenever `root` was among the options, `sun` and `root` correctly for the sun and below-the-soil questions, and produced `root root (2) root root (3) root root (4) root` — matching no option — on a blank white image: the model always produces text, whether or not an answer exists.

In [ ]:
import time

results, seconds = [], []
for entry in questions:
    t0 = time.time()
    results.append(pipe.answer(image, entry['question'], entry['options'], max_new_tokens=max_new_tokens))
    seconds.append(round(time.time() - t0, 2))
print({'device': pipe.device, 'dtype': pipe.dtype, 'seconds_per_question': seconds, 'any_truncated': any(r['truncated'] for r in results), 'unmatched': sum(r['choice_index'] is None for r in results)})
for result in results:
    matched = result['options'][result['choice_index']] if result['choice_index'] is not None else 'NO OPTION MATCHED'
    print(f"Q: {result['question']}  options={result['options']}\n   A: {result['answer']!r} -> {matched}  ({result['new_tokens']} tokens{', TRUNCATED' if result['truncated'] else ''})")
if any(r['truncated'] for r in results):
    print('A budget was exhausted: that answer is incomplete. Raise max_new_tokens (ceiling MAX_NEW_TOKENS) and rerun.')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No accuracy is reported by default: AI2D-style accuracy needs labelled multiple-choice questions on diagrams from the deployment domain, and this repository ships none (the AI2D benchmark is not bundled). When the correct option indices are supplied the report carries `accuracy` (the matched option equals the correct one; an answer that matches no option counts as wrong), `unmatched_rate`, a `chance` baseline (the mean of 1/options over the questions) and one entry per question, with the verdict `sample-sanity`. On the synthetic path those labels are facts **you drew yourself**, so the score proves only that the input contract, header rendering, forward pass, decoding and option matching round-trip — and the six recorded misses show what a wrong answer looks like in the report. On BYOD no correct options are known, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/pix2struct_ai2d_evaluation_report.json`.

In [ ]:
report = evaluation_report(results, correct, sample_kind=sample_kind)
with open('outputs/pix2struct_ai2d_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k not in ('metrics', 'per_question', 'baselines')}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:15} {metric['value']:.3f}  ({metric['estimation']})")
for baseline in report['baselines']:
    print(f"{baseline['id']:15} {baseline['value']:.3f}  ({baseline['note']})")
for entry in report.get('per_question', []):
    print(f"  {'OK  ' if entry['correct'] else 'MISS'}  {entry['question']} -> {entry['prediction']!r} (correct: {entry['correct_option']!r})")
if report['verdict'] == 'not-measurable':
    print('No correct options are known for these questions, so nothing is scored; read the answers against the diagram yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves every result (question, options, prompt, answer, matched index, `new_tokens`, `truncated`, the budget), the evaluation report, the input manifest, the sample identity, digest and correct options, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The question/answer pairs are also written as CSV with explicit `image`, `question`, `options`, `answer`, `choice_index`, `new_tokens`, `truncated` columns, and an annotated PNG shows the diagram with the questions and answers printed in a panel beneath it for visual inspection (the model returns no location, so nothing is drawn on the diagram itself) — a supplement to, not a replacement for, the machine-readable files. No credentials are recorded.

In [ ]:
import csv

panel_height = 30 + 22 * len(results)
annotated = Image.new('RGB', (max(image.width, 900), image.height + panel_height), 'white')
annotated.paste(image.convert('RGB'), (0, 0))
draw = ImageDraw.Draw(annotated)
draw.line([(0, image.height + 1), (annotated.width, image.height + 1)], fill=(120, 120, 120), width=2)
panel_font = ImageFont.load_default(size=14)
for index, result in enumerate(results):
    matched = result['options'][result['choice_index']] if result['choice_index'] is not None else '-'
    draw.text((20, image.height + 12 + 22 * index), f"{result['question']}  ->  {result['answer']} [{matched}]", fill=(40, 90, 220), font=panel_font)
annotated.save('outputs/pix2struct_ai2d_annotated.png')
payload = {
    'predictions': results,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'name': image_name, 'size': list(image.size), 'rgb_sha256': image_sha256, 'questions': questions, 'correct': correct},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/pix2struct_ai2d_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/pix2struct_ai2d_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'question', 'options', 'answer', 'choice_index', 'new_tokens', 'truncated'])
    for result in results:
        writer.writerow([image_name, result['question'], ' | '.join(result['options']), result['answer'], result['choice_index'], result['new_tokens'], result['truncated']])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The answers are the text the model generates after reading a diagram with the question and options printed above it; nothing in the output scores that text, the model returns no location or evidence, and it answers every question — including one about a blank image — with equal fluency, sometimes with text that matches no option. On the drawn plant the `accuracy` in the evaluation report compares the matched options with facts you drew yourself and the verdict is `sample-sanity`, which proves only that the input contract, header rendering, forward pass, decoding and option matching work (the repository's smoke run scored 4/10 against a chance baseline of 0.25, answering `root` to nearly every label question whenever `root` was an option — and `blossom` when the options were synonyms without it); they say nothing about textbook diagrams, photographs, charts, questions that need reasoning across parts, or non-English prompts, and a BYOD result is a single-diagram observation with the verdict `not-measurable`. **The model answers any question about any image** and stops only at end-of-sequence or the token budget: check `truncated` and `choice_index`, and treat a plausible option for an unanswerable question — or a `null` match — as the expected failure mode, not an exception. The answer also depends on the option set itself (removing `root` changed the answer to `leaf`), so the options are part of the request, not a neutral scoring key. The pipeline provides no OCR, no answer localisation, no open-ended answering, no benchmark evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** reorder or replace the options of a label question and watch the answer follow the option set; ask `What color is the flower?` with `pink` among the options (the smoke run said `green`); lower `max_new_tokens` to 1 and watch `truncated` turn true; enable `USE_BYOD` with a textbook diagram you know, type your questions as `question | option | option | …`, then pass your own correct indices to `evaluation_report` to see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/pix2struct-ai2d-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/pix2struct-ai2d-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/pix2struct-ai2d-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google/pix2struct-ai2d-base
- Upstream code: https://github.com/google-research/pix2struct
- Pix2Struct: Screenshot Parsing as Pretraining for Visual Language Understanding (Lee et al., 2022): https://arxiv.org/abs/2210.03347
- A Diagram Is Worth A Dozen Images — the AI2D dataset (Kembhavi et al., 2016): https://arxiv.org/abs/1603.07396